# Practice 17 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime + MultiIndex
A gym check-in DataFrame has been created for you.

1. Convert `date` to datetime. Extract `year` and `month`.
2. Add `days_ago` = days between `date` and `2026-03-15`. Find mean and median with NumPy.
3. Set a `(gym, member)` MultiIndex. Sort it.
4. Retrieve all rows for `'Downtown'`.
5. Use `pd.IndexSlice` to get all `'Carol'` rows across every gym.

In [9]:
# Starter data — don't change this
checkins = pd.DataFrame({
    'gym':      ['Downtown', 'Downtown', 'Uptown', 'Uptown', 'Midtown', 'Midtown', 'Downtown', 'Midtown'],
    'member':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Carol', 'Grace', 'Hiro'],
    'date':     ['2024-02-10', '2024-05-18', '2024-07-22', '2024-09-03',
                 '2024-11-14', '2025-01-08', '2025-03-25', '2025-06-30'],
    'duration': [45, 60, 30, 75, 50, 90, 40, 65],
    'calories': [320, 450, 210, 580, 370, 720, 290, 480],
})

# Your code here
checkins['date'] = pd.to_datetime(checkins['date'])
checkins['year'] = checkins['date'].dt.year
checkins['month'] = checkins['date'].dt.month
checkins['days_ago'] = (pd.to_datetime('2026-03-15') - checkins['date']).dt.days
checkins = checkins.set_index(['gym','member']).sort_index()
checkins.loc['Downtown']
idx = pd.IndexSlice
checkins.loc[idx[:,'Carol'],:]
#checkins



,,date,duration,calories,year,month,days_ago
gym,member,,,,,,
Midtown,Carol,2025-01-08,90,720,2025,1,431
Uptown,Carol,2024-07-22,30,210,2024,7,601


---
## Level 2 — Transformations

### Task 2: .str accessor ✨ New

`.str` is a pandas accessor that lets you apply string operations to every value in a Series at once — no loop needed.

```python
df['col'].str.upper()                       # uppercase every value
df['col'].str.lower()                       # lowercase every value
df['col'].str.contains('word')              # boolean mask — True where 'word' appears
df['col'].str.contains('word', case=False)  # case-insensitive
df['col'].str.split('-')                    # split each string into a list
df['col'].str.split('-').str[-1]            # take the last element after splitting
df['col'].str.len()                         # character count of each string
df['col'].str.replace('old', 'new')         # replace a substring
```

A product catalog DataFrame has been created for you.

1. Add a `name_upper` column: product names in uppercase
2. Filter to rows where `name` contains `'Coffee'` (case-insensitive)
3. Extract the size code from `sku` — split on `'-'` and take the last part. Store as `size`.
4. Add a `name_length` column: number of characters in each product name
5. Replace `'dairy-alt'` with `'dairy'` in the `category` column (update in place)

In [19]:
# Starter data — don't change this
products = pd.DataFrame({
    'name':     ['Organic Oat Milk', 'Dark Roast Coffee', 'Matcha Green Tea',
                 'Earl Grey Tea', 'Oat Biscuits', 'Cold Brew Coffee'],
    'category': ['dairy-alt', 'coffee', 'tea', 'tea', 'snacks', 'coffee'],
    'sku':      ['DA-001-L', 'CO-002-M', 'TE-003-S', 'TE-004-M', 'SN-005-L', 'CO-006-S'],
    'price':    [3.99, 4.49, 3.29, 2.99, 1.99, 4.99],
})

# Your code here

products['name_upper'] = products['name'].str.upper()
c = products[products['name'].str.contains('Coffee')]
products['size'] = products['sku'].str.split('-').str[-1]
products['name_length'] = products['name'].str.len()
products['category'] = products['category'].str.replace('dairy-alt','dairy')
products

,name,category,sku,price,name_upper,size,name_length
0,Organic Oat Milk,dairy,DA-001-L,3.99,ORGANIC OAT MILK,L,16
1,Dark Roast Coffee,coffee,CO-002-M,4.49,DARK ROAST COFFEE,M,17
2,Matcha Green Tea,tea,TE-003-S,3.29,MATCHA GREEN TEA,S,16
3,Earl Grey Tea,tea,TE-004-M,2.99,EARL GREY TEA,M,13
4,Oat Biscuits,snacks,SN-005-L,1.99,OAT BISCUITS,L,12
5,Cold Brew Coffee,coffee,CO-006-S,4.99,COLD BREW COFFEE,S,16


### Task 3: stack() / unstack()
A wide DataFrame of store revenues (stores as index, quarters as columns) has been created for you.

1. Stack to long format — store as `revenue_long` (use `.to_frame('revenue')`)
2. Add `log_revenue` using NumPy
3. Add `high_revenue`: `True` if revenue > 80 (use `np.where`)
4. Unstack back to wide format
5. Find the store with the highest mean revenue using NumPy

In [ ]:
# Starter data — don't change this
np.random.seed(12)
stores = pd.DataFrame({
    'store': ['Oslo', 'Lima', 'Accra', 'Seoul'],
    'Q1':    np.random.randint(40, 120, size=4),
    'Q2':    np.random.randint(40, 120, size=4),
    'Q3':    np.random.randint(40, 120, size=4),
    'Q4':    np.random.randint(40, 120, size=4),
}).set_index('store')

# Your code here
revenue_long = stores.stack().to_frame('revenue')
revenue_long['log_revenue'] = np.log(revenue_long['revenue'])
revenue_long['high_revenue'] = np.where(revenue_long['revenue'] > 80, True, False)
w = revenue_long.unstack()
w
sm = revenue_long.groupby(level='store')['revenue'].agg(np.mean)
sm.idxmax()
#revenue_long

/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_67621/3738929728.py:17: FutureWarning: The provided callable <function mean at 0x7f95a657b430> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  sm = revenue_long.groupby(level='store')['revenue'].agg(np.mean)


'Accra'

### Task 4: .rank() ✨ New

`.rank()` assigns a numeric rank to each value in a Series — useful for leaderboards and within-group comparisons.

```python
s.rank()                    # lowest value → rank 1.0 (default)
s.rank(ascending=False)     # highest value → rank 1.0
s.rank(method='dense')      # no rank gaps on ties: 1, 2, 2, 3  (not 1, 2, 2, 4)

# Rank within groups — use groupby + transform:
df.groupby('group')['col'].rank(ascending=False)
```

A sales rep performance DataFrame has been created for you.

1. Add `sales_rank`: overall rank by `sales`, highest first
2. Add `region_rank`: rank within each `region` by `sales`, highest first
3. Filter to show only the #1-ranked rep per region (where `region_rank == 1`)

In [65]:
# Starter data — don't change this
np.random.seed(33)
reps = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace', 'Hiro'],
    'region': ['North', 'North', 'South', 'South', 'North', 'South', 'North', 'South'],
    'sales':  np.random.randint(50, 200, size=8),
    'calls':  np.random.randint(20, 100, size=8),
})

# Your code here
reps['sales_rank'] = reps['sales'].rank(ascending=False)
reps['region_rank'] = reps.groupby('region')['sales'].rank(ascending=False)
reps[reps['region_rank']==1]

,name,region,sales,calls,sales_rank,region_rank
1,Bob,North,185,65,3.0,1.0
3,Dave,South,196,53,1.0,1.0


---
## Level 3 — Aggregation

### Task 5: pd.merge() + groupby + transform
Two DataFrames have been created for you: `customers` and `orders`.

1. Inner join on `customer_id`
2. Add `city_avg_amount`: mean order amount per city using `transform`
3. Add `above_city_avg`: `True` if `amount` > `city_avg_amount`
4. Find total order amount per `tier` using groupby
5. Find total order amount per `city` using groupby — sort descending

In [32]:
# Starter data — don't change this
np.random.seed(77)
customers = pd.DataFrame({
    'customer_id': [f'C{i:03d}' for i in range(1, 9)],
    'city': ['London', 'Paris', 'Berlin', 'Madrid', 'Rome', 'London', 'Paris', 'Berlin'],
    'tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver', 'Bronze', 'Gold', 'Silver'],
})

orders = pd.DataFrame({
    'order_id':    [f'O{i:02d}' for i in range(1, 13)],
    'customer_id': np.random.choice([f'C{i:03d}' for i in range(1, 9)], size=12),
    'amount':      np.random.randint(50, 500, size=12),
    'month':       np.random.choice(['Jan', 'Feb', 'Mar'], size=12),
})

# Your code here
ij = pd.merge(
    customers, 
    orders,
    how='inner',
    on='customer_id'
)
ij['city_avg_amount'] = ij.groupby('city')['amount'].transform('mean')
ij['above_city_avg'] = ij['amount']>ij['city_avg_amount']
ij.groupby('tier')['amount'].agg('sum')
ij.groupby('city')['amount'].agg('sum').sort_values(ascending=False)

city
Berlin    1273
London    1254
Rome       831
Paris      481
Madrid     424
Name: amount, dtype: int64

### Task 6: pd.concat() + pivot_table
Two quarterly sales DataFrames have been created for you.

1. Concatenate `sales_q1` and `sales_q2`, reset the index. Store as `sales`.
2. Create a pivot table: mean `revenue` by `product` (rows) and `region` (columns)
3. Create another pivot table: total `revenue` by `product` (rows) and `quarter` (columns)
4. From the second pivot table, find the product with the highest total revenue — use `.idxmax()`

In [41]:
# Starter data — don't change this
np.random.seed(88)
sales_q1 = pd.DataFrame({
    'product': np.random.choice(['Laptop', 'Phone', 'Tablet'], size=8),
    'region':  np.random.choice(['North', 'South', 'East'], size=8),
    'revenue': np.random.randint(500, 3000, size=8),
    'quarter': 'Q1',
})

sales_q2 = pd.DataFrame({
    'product': np.random.choice(['Laptop', 'Phone', 'Tablet'], size=8),
    'region':  np.random.choice(['North', 'South', 'East'], size=8),
    'revenue': np.random.randint(500, 3000, size=8),
    'quarter': 'Q2',
})

# Your code here

sales = pd.concat(
    [sales_q1,sales_q2],
    ignore_index=True
)
p = sales.pivot_table(
    index = 'product',
    columns= 'region',
    values= 'revenue'
)
p2 = sales.pivot_table(
    index = 'product',
    columns= 'quarter',
    values= 'revenue'
)
p2
p2t = p2.sum(axis=1)
p2t.idxmax()

'Tablet'

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Three DataFrames have been created for you: `employees`, `projects`, and `timelog`.

1. Convert `hire_date` to datetime. Add `tenure_days` = days between `hire_date` and `2026-03-15`. Find mean and median with NumPy.
2. Inner join `timelog` with `employees` on `emp_id`. Inner join the result with `projects` on `project_id`. Drop nulls.
3. Use `.str` to extract the numeric part of `emp_id` (e.g. `'E03'` → `'03'`). Store as `emp_num`.
4. Add `dept_avg_hours`: mean hours per `dept` using `transform`
5. Bin `hours` into 3 equal-width groups labelled `['Low', 'Medium', 'High']`, store as `workload`. Count rows per `workload` (use `observed=True`).
6. Pivot: total `hours` by `dept` (rows) and `month` (columns) — use `pivot_table` with `aggfunc='sum'`
7. Find the correlation between `hours` and `budget` using NumPy

In [ ]:
# Starter data — don't change this
np.random.seed(42)
n = 12

employees = pd.DataFrame({
    'emp_id':     [f'E{i:02d}' for i in range(1, n + 1)],
    'name':       [f'Employee_{i}' for i in range(1, n + 1)],
    'dept':       np.random.choice(['Eng', 'Sales', 'Design'], size=n),
    'hire_date':  pd.date_range('2020-01-01', periods=n, freq='90D').strftime('%Y-%m-%d'),
})

projects = pd.DataFrame({
    'project_id': [f'P{i:02d}' for i in range(1, 7)],
    'status':     ['Active', 'Completed', 'Active', 'Completed', 'Active', 'Active'],
    'budget':     [50000, 30000, 75000, 20000, 60000, 45000],
})

timelog = pd.DataFrame({
    'emp_id':     np.random.choice([f'E{i:02d}' for i in range(1, n + 1)], size=30),
    'project_id': np.random.choice([f'P{i:02d}' for i in range(1, 7)], size=30),
    'hours':      np.random.randint(5, 40, size=30),
    'month':      np.random.choice(['Jan', 'Feb', 'Mar'], size=30),
})

# Your code here
employees['hire_date'] = pd.to_datetime(employees['hire_date'])
employees['tenure_days'] = (pd.to_datetime('2026-03-15')-employees['hire_date']).dt.days
np.mean(employees['tenure_days'])
np.median(employees['tenure_days'])
ij1 = pd.merge(employees,
               timelog, 
               how='inner',
               on='emp_id')
ij2 = pd.merge(
    ij1,
    projects,
    how='inner',
    on='project_id'
).dropna()
ij2['emp_num'] = ij2['emp_id'].str.extract(r'(\d+)')[0]
ij2['dept_avg_hours'] = ij2.groupby('dept')['hours'].transform('mean')
ij2['workload'] = pd.cut(ij2['hours'],3,labels=['Low','Medium','High'])
ij2.groupby('workload',observed=True).agg('count')
ij2.pivot_table(
    index = 'dept',
    columns = 'month',
    values = 'hours',
    aggfunc = 'sum'
)
np.corrcoef(ij2['hours'],ij2['budget'])


array([[1.        , 0.21276332],
       [0.21276332, 1.        ]])